In [7]:
pip install google-genai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import google.generativeai as genai

# ============================================================
# CARREGAR VARIÁVEIS .ENV
# ============================================================


API_KEY = "COLOQUE_SUA_API_KEY_AQUI" # OBS: O USUARIO DEVE CRIAR E COLOCAR SUA PROPRIA API KEY NO GEMINI

if not API_KEY:
    raise ValueError(
        "API KEY não encontrada. Configure GEMINI_API_KEY no arquivo .env"
    )

# ============================================================
# CONFIGURAÇÃO GEMINI
# ============================================================

genai.configure(api_key=API_KEY)

modelo = genai.GenerativeModel("gemini-2.5-flash")

# ============================================================
# DEFINIÇÃO DE PERSONAS
# ============================================================

PERSONAS = {
    "1": {
        "nome": "Operador Comercial",
        "descricao": """
Você é um especialista comercial da GoodWe.

Seu foco é:
- Explicar produtos
- Explicar benefícios dos inversores
- Explicar garantias
- Explicar monitoramento
- Explicar diferenciais técnicos
- Auxiliar parceiros e vendedores

Sempre responda de forma profissional e objetiva.
"""
    },

    "2": {
        "nome": "Síndico",
        "descricao": """
Você é um especialista GoodWe para condomínios.

Seu foco é:
- Explicar economia de energia
- Explicar funcionamento de sistema solar
- Explicar manutenção
- Explicar segurança
- Explicar monitoramento

Use linguagem simples e clara.
"""
    },

    "3": {
        "nome": "Morador",
        "descricao": """
Você é um assistente GoodWe para clientes residenciais.

Seu foco é:
- Explicar geração solar
- Explicar aplicativo de monitoramento
- Explicar economia
- Explicar dúvidas básicas
- Explicar alertas do sistema

Use linguagem amigável e simples.
"""
    },

    "4": {
        "nome": "Técnico",
        "descricao": """
Você é um especialista técnico da GoodWe.

Seu foco é:
- Diagnóstico de falhas
- Alarmes de inversores
- Configuração
- Comunicação Wi-Fi/LAN
- Startup
- Paralelismo
- Manutenção técnica

Use linguagem técnica e detalhada.
"""
    }
}

# ============================================================
# FAQ - PERGUNTAS ESPERADAS
# ============================================================

FAQ = {
    "1": [
        {
            "pergunta": "Quais são os diferenciais dos inversores GoodWe?",
            "resposta": """
Os inversores GoodWe possuem alta eficiência,
monitoramento remoto, proteções avançadas
e excelente custo-benefício.
"""
        },

        {
            "pergunta": "Como funciona o monitoramento?",
            "resposta": """
O monitoramento é feito pelo SEMS Portal,
permitindo acompanhar geração e alertas em tempo real.
"""
        },

        {
            "pergunta": "Qual é a garantia?",
            "resposta": """
A garantia varia conforme o modelo,
com possibilidade de extensão.
"""
        },

        {
            "pergunta": "Os inversores possuem proteção?",
            "resposta": """
Sim. Possuem proteções elétricas,
contra surtos e sobretemperatura.
"""
        },

        {
            "pergunta": "É possível integrar baterias?",
            "resposta": """
Sim. Alguns modelos híbridos GoodWe
são compatíveis com baterias.
"""
        }
    ],

    "2": [
        {
            "pergunta": "O condomínio economiza energia?",
            "resposta": """
Sim. A energia solar reduz significativamente
os custos das áreas comuns.
"""
        },

        {
            "pergunta": "O sistema precisa de manutenção?",
            "resposta": """
A manutenção é simples e preventiva,
incluindo limpeza e inspeções periódicas.
"""
        },

        {
            "pergunta": "Como acompanho a geração?",
            "resposta": """
O acompanhamento pode ser feito
por aplicativo e plataforma online.
"""
        },

        {
            "pergunta": "O sistema é seguro?",
            "resposta": """
Sim. Os sistemas possuem proteções
e seguem normas técnicas.
"""
        },

        {
            "pergunta": "Funciona em dias nublados?",
            "resposta": """
Sim, porém com menor eficiência.
"""
        }
    ],

    "3": [
        {
            "pergunta": "Como vejo minha geração solar?",
            "resposta": """
Você pode acompanhar pelo aplicativo SEMS Portal.
"""
        },

        {
            "pergunta": "Por que minha geração caiu?",
            "resposta": """
Pode ocorrer devido ao clima,
sujeira nos módulos ou sombreamento.
"""
        },

        {
            "pergunta": "O sistema funciona à noite?",
            "resposta": """
Não. Apenas sistemas com bateria
conseguem fornecer energia à noite.
"""
        },

        {
            "pergunta": "O aplicativo mostra consumo?",
            "resposta": """
Dependendo da configuração,
é possível visualizar geração e consumo.
"""
        },

        {
            "pergunta": "Preciso desligar o inversor?",
            "resposta": """
Não. O inversor deve permanecer ligado,
exceto em manutenção.
"""
        }
    ],

    "4": [
        {
            "pergunta": "O inversor está sem comunicação.",
            "resposta": """
Verifique:
- Wi-Fi
- Datalogger
- Internet
- LEDs de comunicação
"""
        },

        {
            "pergunta": "Erro de isolação.",
            "resposta": """
Verifique:
- Cabos DC
- Conectores MC4
- Umidade
- String box
"""
        },

        {
            "pergunta": "O inversor não conecta ao SEMS.",
            "resposta": """
Verifique internet, número de série
e status do datalogger.
"""
        },

        {
            "pergunta": "O inversor não gera energia.",
            "resposta": """
Verifique tensão DC,
rede AC e disjuntores.
"""
        },

        {
            "pergunta": "Existe atualização de firmware?",
            "resposta": """
Sim. Algumas atualizações
podem ser feitas remotamente.
"""
        }
    ]
}

# ============================================================
# MENU
# ============================================================

def mostrar_menu():

    print("\n" + "=" * 60)
    print("      CHATBOT DE SUPORTE GOODWE - GEMINI")
    print("=" * 60)

    print("\nESCOLHA A PERSONA:\n")

    print("1 - Operador Comercial")
    print("2 - Síndico")
    print("3 - Morador")
    print("4 - Técnico")


# ============================================================
# EXIBIR FAQ
# ============================================================

def mostrar_faq(opcao):

    print("\n" + "=" * 60)
    print("PERGUNTAS ESPERADAS E RESPOSTAS IDEAIS")
    print("=" * 60)

    for i, item in enumerate(FAQ[opcao], start=1):

        print(f"\n{i}. PERGUNTA:")
        print(item["pergunta"])

        print("\nRESPOSTA IDEAL:")
        print(item["resposta"])

        print("-" * 60)


# ============================================================
# PROMPT DO SISTEMA
# ============================================================

def criar_system_prompt(persona):

    return f"""
Você é um chatbot oficial de suporte GoodWe.

Persona:
{persona['nome']}

Descrição:
{persona['descricao']}

Regras:
- Responder em português do Brasil
- Ser educado
- Ser claro
- Não inventar informações
- Ser técnico quando necessário
- Explicar de forma objetiva
"""


# ============================================================
# CHATBOT
# ============================================================

def iniciar_chat(system_prompt):

    print("\n" + "=" * 60)
    print("CHATBOT INICIADO")
    print("Digite 'sair' para encerrar")
    print("=" * 60)

    historico = []

    while True:

        pergunta = input("\nVocê: ")

        if pergunta.lower() == "sair":
            print("\nChat encerrado.")
            break

        historico.append(f"Usuário: {pergunta}")

        contexto = "\n".join(historico)

        prompt_final = f"""
{system_prompt}

Histórico:
{contexto}

Usuário:
{pergunta}
"""

        try:

            resposta = modelo.generate_content(prompt_final)

            texto = resposta.text

            historico.append(f"Assistente: {texto}")

            print(f"\nGoodWe IA:\n{texto}")

        except Exception as erro:

            print("\nErro ao consultar Gemini:")
            print(erro)


# ============================================================
# MAIN
# ============================================================

def main():

    mostrar_menu()

    opcao = input("\nDigite a opção desejada: ")

    if opcao not in PERSONAS:
        print("\nOpção inválida.")
        return

    persona = PERSONAS[opcao]

    print(f"\nPersona selecionada: {persona['nome']}")

    mostrar_faq(opcao)

    system_prompt = criar_system_prompt(persona)

    iniciar_chat(system_prompt)


# ============================================================
# EXECUÇÃO
# ============================================================

if __name__ == "__main__":
    main()


      CHATBOT DE SUPORTE GOODWE - GEMINI

ESCOLHA A PERSONA:

1 - Operador Comercial
2 - Síndico
3 - Morador
4 - Técnico

Persona selecionada: Morador

PERGUNTAS ESPERADAS E RESPOSTAS IDEAIS

1. PERGUNTA:
Como vejo minha geração solar?

RESPOSTA IDEAL:

Você pode acompanhar pelo aplicativo SEMS Portal.

------------------------------------------------------------

2. PERGUNTA:
Por que minha geração caiu?

RESPOSTA IDEAL:

Pode ocorrer devido ao clima,
sujeira nos módulos ou sombreamento.

------------------------------------------------------------

3. PERGUNTA:
O sistema funciona à noite?

RESPOSTA IDEAL:

Não. Apenas sistemas com bateria
conseguem fornecer energia à noite.

------------------------------------------------------------

4. PERGUNTA:
O aplicativo mostra consumo?

RESPOSTA IDEAL:

Dependendo da configuração,
é possível visualizar geração e consumo.

------------------------------------------------------------

5. PERGUNTA:
Preciso desligar o inversor?

RESPOSTA IDEAL:

